# LifeMate — finetune QLoRA trên Kaggle (2× T4)

Notebook này **chỉ** huấn luyện adapter. Suy luận và chấm điểm nằm ở
`lifemate_kaggle_v2.ipynb`, chạy sau khi có adapter.

Trước đây nó chỉ tồn tại trên Kaggle, không có trong repo — nên hai lượt chết
(4 giờ 12 phút và 5 phút) không để lại vết nào để xem lại. Bản này dựng lại từ
log của lượt chạy, **không phải bản sao** notebook cũ, nên hãy đối chiếu phần
`ARGS` với thứ bạn thực sự muốn chạy.

## Cần gì trong Input
- Dataset `lifemate-sft` chứa: `train_sft.py`, `sft.train.jsonl`, `sft.eval.jsonl`
- Accelerator: **GPU T4 x2**, Internet **On** (để tải trọng số lần đầu)

## Lượt chạy
320 mẫu, `grad-accum 4` → 80 bước. Đo được 298 giây/bước → **~6.6 giờ**.
`--save-steps 10` nên nếu đứt thì mất tối đa ~50 phút; chạy lại y nguyên lệnh
này, `--resume` tự tiếp từ checkpoint mới nhất.

In [ ]:
# 1. Phien ban, duong dan, va kiem tra Input truoc khi lam gi ton thoi gian
import sys, subprocess, importlib.metadata as md
from pathlib import Path

for pkg in ("transformers", "accelerate", "peft", "bitsandbytes", "torch"):
    try:
        print(f"{pkg:14} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14} THIEU")

# Do bang glob, KHONG duong dan cung: dataset mount o /kaggle/input/datasets/<user>/<slug>/
# chu khong phai /kaggle/input/<slug>/. Duong cung sai tung lam preflight lay mau tu ca split
# va lam nhiem diem -- xem EXPERIMENTS.md.
def find_one(pattern: str) -> Path:
    hits = sorted(Path("/kaggle/input").glob(pattern))
    if not hits:
        raise FileNotFoundError(f"khong thay {pattern} trong /kaggle/input -- da Add Data chua?")
    return hits[0]

SCRIPT = find_one("**/train_sft.py")
TRAIN  = find_one("**/sft.train.jsonl")
EVAL   = find_one("**/sft.eval.jsonl")
OUT    = Path("/kaggle/working/lora_V4")

print()
for name, p in (("script", SCRIPT), ("train", TRAIN), ("eval", EVAL)):
    print(f"{name:7} {p}")

# In ngay ban train_sft.py dang dung. Hai luot truoc chet vi script tren dataset KHAC voi
# script trong repo; mot dong hash o day la cach duy nhat doc log ma biet minh chay ban nao.
import hashlib
print()
print(f"sha256(train_sft.py)[:16] = {hashlib.sha256(SCRIPT.read_bytes()).hexdigest()[:16]}")
print(f"so dong = {len(SCRIPT.read_text(encoding='utf-8').splitlines())}")

In [ ]:
# 2. Cai lai bitsandbytes -- Kaggle da bo no khoi image
#
# Khong co no thi optim roi ve adamw_torch, va trang thai optimizer fp32 cho 66M tham so
# LoRA ton them ~400 MB. Luot preflight vua roi chi con 682 MiB trong tren 14.56 GiB, nen
# 400 MB la khoan dang lay lai. Cai that bai thi KHONG sao: train_sft.py tu roi ve
# adamw_torch va in ra dong canh bao.
!pip install -q bitsandbytes
import importlib.metadata as md
try:
    print("bitsandbytes", md.version("bitsandbytes"))
except md.PackageNotFoundError:
    print("bitsandbytes VAN THIEU -> se dung adamw_torch, +~400 MB")

In [ ]:
# 2. Tai trong so (chi lan dau; sau do nam trong /root/.cache nen chay lai la mien phi)
from huggingface_hub import snapshot_download

LLM_ID  = "Qwen/Qwen3-4B-Instruct-2507"
LLM_REV = "cdbee75f17c01a7cc42f958dc650907174af0554"
# Ban INSTRUCT, khong phai Thinking: quy dao gold khong co khoi <think> nao, va adapter hoc tren
# trong so Instruct khong nap duoc lenlen trong so Thinking. Doi cho nay la doi ca huong du an.

MODEL = snapshot_download(LLM_ID, revision=LLM_REV,
                          allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.txt"])
print("model :", MODEL)

In [ ]:
# PREFLIGHT -- 2 mau, 2 buoc optimizer thuc su. Khoang 10 phut.
#
# Vi sao bat buoc: ca ba luot chet truoc do deu NAP MODEL THANH CONG. Loi nam trong
# compute_loss va trong backward, ma ca hai chi chay tu buoc 1. Nap model xong khong
# chung minh gi. Den gio chang nay da chan duoc 3 lan: warmup_ratio bi bo o
# transformers 5.16.1, bitsandbytes bi go khoi image Kaggle, va OOM 1024 MiB/khoi.
PRE = [
    "--train", str(TRAIN), "--eval", str(EVAL),
    "--model", MODEL,
    "--out", "/kaggle/working/preflight",
    "--attn", "block", "--quant", "fp16", "--device", "single",
    "--limit", "2", "--eval-limit", "2",
    "--grad-accum", "1",      # 2 mau / 1 = 2 buoc optimizer THAT
    "--epochs", "1",
    "--query-block", "256",   # 1024 doi 1024 MiB/khoi va OOM; 256 -> 256 MiB, CUNG ket qua
    "--block-checkpoint",    # dinh = MOT khoi; khong co co nay thi autograd giu MOI khoi
    "--mem-trace",           # in bo nho tung moc cua 3 buoc dau
    "--max-seq-len", "8192",  # phai GIONG luot that: VRAM dinh phu thuoc cho nay
    "--cache", "/tmp/sft.tok",
]
proc = subprocess.run([sys.executable, str(SCRIPT)] + PRE)
print("returncode =", proc.returncode)
if proc.returncode != 0:
    raise SystemExit("preflight that bai -- DUNG. Sua truoc khi tieu 6.6 gio quota.")

# Phai doc tu log o tren truoc khi sang cell sau:
#   1. "ghim n_gpu=1: khong boc DataParallel"   -> dataset la ban MOI. Khong thay = ban CU, dung lai.
#   2. "dinh mot khoi ~ 256 MiB"                -> --query-block da an.
#   3. "VRAM dinh: ... GiB"                     -> phai duoi 14.56, va con bien de chay 320 mau.
#   4. loss cua buoc 1 va 2 la so thuc, khong nan -> compute_loss chay dung.

In [ ]:
# 3. Huan luyen. Moi co o day deu co mot luot chay chet dang sau no.
ARGS = [
    "--train", str(TRAIN), "--eval", str(EVAL),
    "--model", MODEL,
    "--out", str(OUT),
    "--attn", "block",        # khong dung ma tran (1,32,L,L) tren sm_75
    "--quant", "fp16",        # 7.49 GiB, vua MOT card T4 (14.56 GiB)
    "--device", "single",     # xem ghi chu ben duoi
    "--limit", "320", "--eval-limit", "40",
    "--grad-accum", "4",      # 320/4 = 80 buoc
    "--save-steps", "10",     # luu theo BUOC: mot luot chet o buoc 46/80 sau 4h12 da mat trang
    "--resume",               # chay lai y nguyen lenh nay thi tu tiep tu checkpoint moi nhat
    "--epochs", "1",
    "--query-block", "256",   # xem ghi chu o cell preflight
    "--block-checkpoint",    # dinh = MOT khoi; khong co co nay thi autograd giu MOI khoi
    "--mem-trace",           # in bo nho tung moc cua 3 buoc dau
    "--max-seq-len", "8192",  # 8192 giu 100% hoi thoai (dai nhat 8181 token); 4096 chi giu 18.8%
    "--cache", "/tmp/sft.tok",
]
# --device single: dat trong so len mot card de bo han hook chuyen tensor cua accelerate, cho no
# illegal memory access o buoc 46/80. Nhung Trainer KHONG doc device_map -- no doc
# torch.cuda.device_count(), thay 2 card, dat n_gpu=2 roi boc nn.DataParallel, lam luot sau chet
# ngay buoc 0 voi "khong tim thay lm_head tren DataParallel". train_sft.py gio ghim n_gpu=1;
# log phai in dong "ghim n_gpu=1: khong boc DataParallel", neu khong thay thi script tren dataset
# la ban CU -- dung de no chay tiep, cap nhat dataset truoc.
# Khong can --vram-fraction: chot chan VRAM tu bo qua tren card >= 8 GiB.

print(" ".join(["python", SCRIPT.name] + ARGS), flush=True)
proc = subprocess.run([sys.executable, str(SCRIPT)] + ARGS)
print("returncode =", proc.returncode)
if proc.returncode != 0:
    raise SystemExit("train_sft.py that bai -- doc traceback o tren truoc khi chay lai")

In [ ]:
# 4. Adapter da ra chua, va nang bao nhieu
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1024**2:9.2f} MB  {p.relative_to(OUT)}")
print("Tai ca thu muc lora_V4 ve, roi dat vao mot Kaggle Dataset moi de"
      " lifemate_kaggle_v2.ipynb tro ADAPTER_DIR vao do.")